In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import pandas as pd

import shutil
import os

from datasets import Dataset
from huggingface_hub import login

import utils


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0



In [ ]:
data_dir = '/home/shay/PycharmProjects/sensim/data'
base_model_name = 'OMRIDRORI/mbert-tibetan-continual-wylie-final'

In [ ]:

hf_gated_omri = None

login(token=hf_gated_omri)

In [ ]:

# --- Setup ---
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Use a margin of 1.0
loss_fn = torch.nn.MarginRankingLoss(margin=1.0)
model.train()

# --- Example Batch (2 pairs) ---
# Pair 1: (A, B)
# Pair 2: (A, C)
better_sentences = ["Sentence A (Best)", "Sentence A (Best)"]
worse_sentences = ["Sentence B (Other)", "Sentence C (Other)"]

# Tokenize
inputs_better = tokenizer(better_sentences, padding=True, truncation=True, return_tensors="pt")
inputs_worse = tokenizer(worse_sentences, padding=True, truncation=True, return_tensors="pt")

# --- Training Step ---
optimizer.zero_grad()

# Get scores for the "better" sentences
outputs_better = model(**inputs_better)
scores_better = outputs_better.logits.squeeze() # Shape: [batch_size]

# Get scores for the "worse" sentences
outputs_worse = model(**inputs_worse)
scores_worse = outputs_worse.logits.squeeze() # Shape: [batch_size]

# Target tensor: A tensor of 1s, telling the loss we want score_better > score_worse
target = torch.ones_like(scores_better)

# Calculate loss
loss = loss_fn(scores_better, scores_worse, target)
loss.backward()
optimizer.step()